# Libraries

In [1]:
#!pip install -q torch==2.9.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip -q install "lm_eval[hf]"
!pip -q install --upgrade transformers
!pip -q install langdetect --break-system-packages #for If-eval Task

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 83.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 100.7 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.8 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


In [2]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient

# Global Variables

In [3]:
MODEL_NAME       = "Llama-3.2-3B-Instruct"
MODEL_PRETRAINED = "meta-llama/Llama-3.2-3B-Instruct"
SEED             = 42

os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [4]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    max_gen_toks:      Optional[int] = None    

# Functions

In [5]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [6]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface login

In [7]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Evaluations

**Configurations**

In [ ]:
# Task definition
TASKS = [
    # Math
     Task("gsm8k",        "gsm8k_cot",                 "math",                num_fewshot=8, batch_size=12,  max_gen_toks=1024),

    # Reasoning
     Task("arc_challenge","arc_challenge",              "reasoning",           batch_size=16),

    # Instruction Following
     Task("ifeval",       "ifeval",                    "instruction_following",batch_size=14,  max_gen_toks=1024),

    # Perplexity
     Task("wikitext",     "wikitext",                  "perplexity",          batch_size=1,  apply_chat_template=False),

    # Knowledge
     Task("mmlu",         "mmlu",                      "knowledge",           num_fewshot=5, batch_size=2,  limit=1400),
]


# Model args
MODEL_ARGS = ",".join([
    f"pretrained={MODEL_PRETRAINED}",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
    "trust_remote_code=True",
    #"enable_thinking=False", #For qwen models only
])

results_dir = f"evaluation_results/{MODEL_NAME}"

**Evaluation loop**

In [9]:
for t in TASKS:
    print(f"\n[{t.category.upper()}] {t.key}")

    cmd = [
        "lm_eval",
        "--model",       "hf",
        "--model_args",  MODEL_ARGS,
        "--tasks",       t.name,
        "--num_fewshot", str(t.num_fewshot),
        "--batch_size",  str(t.batch_size),
        "--seed",        str(SEED),
        "--verbosity", "WARNING",
        "--output_path", f"{results_dir}/{t.key}",
    ]

    if t.limit is not None:
        cmd += ["--limit", str(t.limit)]
        
    if t.apply_chat_template:
            cmd.append("--apply_chat_template")
        
    if t.max_gen_toks is not None:                             
        cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]

    subprocess.run(cmd, check=True)

    torch.cuda.empty_cache()
    gc.collect()
    print("Done")


[KNOWLEDGE] mmlu


2026-04-06:07:16:10 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-04-06:07:16:10 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-06:07:16:17 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 2404.72 examples/s]
2026-04-06:07:18:32 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_abstract_algebra from None to 5
2026-04-06:07:18:32 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_anatomy from None to 5
2026-04-06:07:18:32 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_astronomy from None to 5
2026-04-06:07:18:32 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_biology from None to 5
2026-04-06:07:18:32 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_chemistry from None to 5
2026-04-06:07:18:32 WARNING  [evaluato

hf ({'pretrained': 'meta-llama/Llama-3.2-3B-Instruct', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({}), limit: 1400.0, num_fewshot: 5, batch_size: 2
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.6199|±  |0.0039|
| - humanities                          |      2|none  |      |acc   |↑  |0.5922|±  |0.0070|
|  - formal_logic                       |      1|none  |     5|acc   |↑  |0.4286|±  |0.0443|
|  - high_school_european_history       |      1|none  |     5|acc   |↑  |0.7455|±  |0.0340|
|  - high_school_us_history             |      1|none  |     5|acc   |↑  |0.7745|±  |0.0293|
|  - high_school_world_history          |      1|none  |     5|acc   |↑  |0.8101|±  |0.0255|
|  - international_law                  |      1|none  |     5|acc   |↑  |0

# Save reports

In [10]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)